# N112 · 状态压缩DP与集合状态

**目标：** 把已选择对象集合编码成充分状态。

**先修：** N098, N124, N101, N064。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** mask；最后位置；子集转移；访问状态图；状态数与转移数。

## 从问题到算法

把小集合编码成mask后，“已经覆盖哪些技能”“已经访问哪些顶点”变成可哈希状态。状态必须保留会影响未来的全部信息：旅行问题还需要最后位置；贴纸可把剩余字符计数压成元组。指数复杂度仍然存在，压缩只是让它可控而非变成多项式。

## 最小实现

**本章接口：** `smallest_sufficient_team(skills, people)`、`min_stickers(stickers, target)`、`hamiltonian_path_dp(cost)`

In [1]:
from functools import cache
from collections import Counter

def smallest_sufficient_team(skills,people):
    pos={s:i for i,s in enumerate(skills)}; full=(1<<len(skills))-1; dp={0:()}
    for i,person in enumerate(people):
        mask=0
        for s in person:
            if s in pos: mask|=1<<pos[s]
        for covered,team in list(dp.items()):
            new=covered|mask
            if new not in dp or len(team)+1<len(dp[new]): dp[new]=team+(i,)
    return list(dp[full]) if full in dp else None

def min_stickers(stickers,target):
    letters=sorted(set(target)); wanted=Counter(target); vectors=[tuple(Counter(s)[c] for c in letters) for s in stickers]
    @cache
    def f(state):
        if not any(state): return 0
        first=next(i for i,x in enumerate(state) if x); best=float('inf')
        for v in vectors:
            if not v[first]: continue
            new=tuple(max(0,x-y) for x,y in zip(state,v)); best=min(best,1+f(new))
        return best
    answer=f(tuple(wanted[c] for c in letters))
    return -1 if answer==float('inf') else answer

def hamiltonian_path_dp(cost):
    n=len(cost)
    if not n: return 0
    if any(len(row)!=n for row in cost): raise ValueError('square cost matrix required')
    dp=[[float('inf')]*n for _ in range(1<<n)]
    for u in range(n): dp[1<<u][u]=0
    for mask in range(1<<n):
        for u in range(n):
            if not mask>>u&1: continue
            for v in range(n):
                if not mask>>v&1: dp[mask|1<<v][v]=min(dp[mask|1<<v][v],dp[mask][u]+cost[u][v])
    return min(dp[-1])

## 正确性、前提与复杂度

团队每人只处理一轮，读取快照防止本轮重复选择；对同一个覆盖mask，人数更多的方案被支配。贴纸选取顺序无关，可强制下一张覆盖首个未满足字符来剪枝；每次该坐标严格减少，递归终止。Hamiltonian状态按最后一条边追加一个未访问点，既无重复又覆盖所有路径。

**代价：** 团队O(P·2^S·S)保守时间（含元组复制），O(S·2^S)空间；贴纸状态数至多∏(字符需求数+1)；Hamiltonian O(n²2^n)时间O(n2^n)空间。缺失边用+∞，无可行Hamiltonian路径返回+∞。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

skills=['python','sql','js']; people=[['python'],['sql'],['sql','js']]
show_table(['人','技能','掩码'],[(i,p,format(sum(1<<skills.index(s) for s in set(p)),'03b')) for i,p in enumerate(people)])
print('最小团队',smallest_sufficient_team(skills,people))

人,技能,掩码
0,['python'],001
1,['sql'],010
2,"['sql', 'js']",110


最小团队 [0, 2]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert smallest_sufficient_team(['a'],[[],['b']]) is None
assert smallest_sufficient_team([],[])==[]
assert min_stickers(['with','example','science'],'thehat')==3
assert min_stickers(['notice','possible'],'basicbasic')==-1
from itertools import permutations
from random import Random
rng=Random(112)
for n in range(1,8):
    cost=[[rng.randrange(-2,8) for _ in range(n)] for _ in range(n)]
    assert hamiltonian_path_dp(cost)==min(sum(cost[u][v] for u,v in zip(p,p[1:])) for p in permutations(range(n)))
for _ in range(100):
    skills=['a','b','c']; people=[[s for s in skills if rng.randrange(2)] for _ in range(6)]
    team=smallest_sufficient_team(skills,people)
    valid=[mask for mask in range(1<<6) if set().union(*(set(people[i]) for i in range(6) if mask>>i&1))==set(skills)]
    if valid:
        assert len(team)==min(x.bit_count() for x in valid)
        assert set().union(*(set(people[i]) for i in team))==set(skills)
        assert len(team)==len(set(team))
    else: assert team is None
print('N112: 所有本章断言通过')

N112: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 为最小团队恢复人员索引。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 区分O(2^n)状态和可能更多的转移。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 464. Can I Win（canonical）
- 691. Stickers to Spell Word（canonical）
- 1125. Smallest Sufficient Team（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。